# Class Lab: getting started with PySpark

# First steps in PySpark

In [ ]:
from pyspark.sql import SparkSession

# create a Spark session
spark = SparkSession.builder.appName("PySpark_Get_Started").getOrCreate()

In [ ]:
# perform operations using the SparkSession showed in 'Spark UI
spark

In [ ]:
# Create a DataFrame
data = [("apple", 5), ("kiwi", 1), ("banana", 3), ("apple", 3), ("kiwi", 7)]
df = spark.createDataFrame(data, ["Fruit", "Amount"])
# df.show() # text mode table
pretty_df = df.toPandas() # well-formatted data frame by pandas style
pretty_df

In [ ]:
# shut down the current active SparkSession
spark.stop()

# Leverage RDD (Resilient Distributed Dataset)

## RDD creation

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("RDD_demo").getOrCreate()
numbers = [1, 2, 3, 4, 5]
# create a RDD from a list
rdd = spark.sparkContext.parallelize(numbers) 

# collect action: retrieve all elements of the RDD
rdd.collect()

In [ ]:
# create RDD from a list of tuples
fruits = [("apple", 5), ("kiwi", 1), ("banana", 3), ("apple", 3), ("kiwi", 7)]
fruits_rdd = spark.sparkContext.parallelize(fruits)
print("All elements of the fruits rdd", fruits_rdd.collect())


## RDD operations: Actions

In [ ]:
# Count action: count the number of elements in the RDD
fruits_rdd.count()

In [ ]:
# First action: retrieve the first element of the RDD
fruits_rdd.first()

In [ ]:
# Take action: retrieve the first n elements of the RDD
fruits_rdd.take(2)


In [ ]:
# Foreach action: apply a function to each element of the RDD
fruits_rdd.foreach(lambda x: print(x))


## RDD operations: Transformations

In [ ]:
# Map transformation: apply a function to each element of the RDD
mapped_fruits_rdd = fruits_rdd.map(lambda x: (x[0].upper(), x[1]))
mapped_fruits_rdd.collect()

In [ ]:
# Filter transformation: select elements that meet a certain condition
filtered_fruits_rdd = fruits_rdd.filter(lambda x: x[1] >= 5)
filtered_fruits_rdd.collect()

In [ ]:
# ReduceByKey transformation: aggregate values by key
reduced_fruits_rdd = fruits_rdd.reduceByKey(lambda x, y: x + y)
reduced_fruits_rdd.collect()

In [ ]:
# SortBy transformation: sort the RDD by key
sorted_fruits_rdd = fruits_rdd.sortBy(lambda x: x[1], ascending=False)
sorted_fruits_rdd.collect()

## Read / Write RDDs from  / to text file

In [ ]:
# Save action: write the RDD to a text file
fruits_rdd.saveAsTextFile("../txt/fruits.txt")

In [ ]:
# Create RDD from a text file
rdd = spark.sparkContext.textFile("../txt/fruits.txt")
rdd.collect()

In [ ]:
# Word count application by RDD

two_towers_rdd = spark.sparkContext.textFile("../txt/雙城記.txt")
word_counts = two_towers_rdd.flatMap(lambda line: line.split(" ")) \
                       .map(lambda word: (word, 1)) \
                       .reduceByKey(lambda a, b: a + b)
word_counts.collect()

In [ ]:
spark.stop()  # shut down the current active SparkSession

# Leverage DataFrame

In [ ]:
from pyspark.sql import SparkSession
# from pyspark.sql.functions import desc

spark = SparkSession.builder.appName("DataFrame_demo").getOrCreate()

In [ ]:
# load data into DataFrame

data_file_path = "../txt/stocks.txt"
df = spark.read.csv(data_file_path, header=True, inferSchema=True)

# Show the first few rows of the DataFrame
df.printSchema()
df.show(10)

In [ ]:
# select specific columns
selected_columns = df.select("id", "name", "price")
selected_columns.show(10)

In [ ]:
# filter rows based on a condition
filtered_df = df.filter(df["price"] > 100)
filtered_df.show(10)

In [ ]:
# GroupBy and Aggregation
grouped_df = df.groupBy("category").agg({"quantity": "sum", "price": "avg"})
grouped_df.show()

In [ ]:
# Join with another DataFrame
other_data = [("Food", "1"), ("Sports", "2"), ("Electronics", "3"), ("Clothing", "4"), ("Furniture", "5"), ("Accessories", "6")]
other_columns = ["category", "typy_id"]
other_df = spark.createDataFrame(other_data, other_columns)

joined_df = df.join(other_df, df["category"] == other_df["category"], "inner")
joined_df.show()

In [ ]:
# sort by a column
sorted_df = df.orderBy("price", ascending=False)
sorted_df.show(10)

In [ ]:
# sort by multiple columns
sorted_df = df.orderBy(["category", "id"], ascending=[True, False])
sorted_df.show(10)

In [ ]:
# get distinct rows
distinct_rows = df.select("category").distinct()
distinct_rows.show()

In [ ]:
# drop columns
dropped_columns = df.drop("category", "name")
dropped_columns.show(10)

In [ ]:
# add new calculated columns
df_with_new_column = df.withColumn("revenue", df["price"] * df["quantity"])
df_with_new_column.show(10)

In [ ]:
# rename columns with alias
renamed_columns = df.withColumnRenamed("id", "product_id") \
                    .withColumnRenamed("name", "product_name") \
                    .withColumnRenamed("price", "product_price")
renamed_columns.show(10)

In [ ]:
spark.stop()

# Leverage Spark SQL

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SparkSQL_demo").getOrCreate()

In [ ]:
# load data into DataFrame
data_file_path = "../txt/persons.csv"
df = spark.read.csv(data_file_path, header=True, inferSchema=True)

# Show the first few rows of the DataFrame
df.printSchema()
df.show(10)

In [27]:
# register DataFrame as a temporary view
df.createOrReplaceTempView("persons")

In [ ]:
# write SQL queries
result = spark.sql("SELECT * FROM persons WHERE age > 30")
result.show()

In [ ]:
avg_salary_by_gender = spark.sql("""
    SELECT gender, AVG(salary) as avg_salary
    FROM persons
    GROUP BY gender
""")
avg_salary_by_gender.show()

In [28]:
# check if a temporary view exists
if spark.catalog.tableExists("persons"):
    print("Temporary view 'persons' exists.")
else:
    print("Temporary view 'persons' does not exist.")


Temporary view 'persons' exists.


In [29]:
# drop a temporary view
spark.catalog.dropTempView("persons")


True